# Imports & Setup

In [ ]:
# @title Imports
# hide noisy output from pip install
%%capture

!pip install bert_score

import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score as bert_score
from google.colab import drive

## Setup

Due to the resource intensive model training, this notebook is intended to be run in Google Colab using an appropriate runtime.

To import the files that are required for the code below (e.g. training data files), copy them into your Google Drive and connect it to this notebook.

In [ ]:
# Input data files are stored on Google Drive to make them usable within Colab
# Mount Google Drive to access data
drive.mount("/content/drive")

DRIVE_BASE_PATH = "Please enter path to data folder" #@param

In [ ]:
# Ensures reproducibility of data generation
RANDOM_SEED = 42   # @param

# Set seeds for full reproducibility
def set_seed(seed=RANDOM_SEED):
  """ Set all random seeds for reproducibility."""
  random.seed(RANDOM_SEED)
  np.random.seed(RANDOM_SEED)
  torch.manual_seed(RANDOM_SEED)
  torch.cuda.manual_seed_all(RANDOM_SEED)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

set_seed()

# Pairwise Cosine Similarity

In [ ]:
def compute_cosine_similarity(file_path):
    # Load CSV
    df = pd.read_csv(file_path)

    # Ensure the necessary columns exist
    if not all(col in df.columns for col in ["Answer 1", "Answer 2", "Answer 3"]):
        print(f"Skipping {file_path}, required columns not found!")
        return None

    # Convert NaN values to empty strings (prevents errors)
    df[["Answer 1", "Answer 2", "Answer 3"]] = df[["Answer 1", "Answer 2", "Answer 3"]].fillna("")

    # Initialize TF-IDF Vectorizer
    vectorizer = TfidfVectorizer()

    # Compute TF-IDF representations
    tfidf_matrix = vectorizer.fit_transform(df["Answer 1"].tolist() +
                                            df["Answer 2"].tolist() +
                                            df["Answer 3"].tolist())

    # Split into respective answer groups
    n = len(df)  # Number of rows
    tfidf_A1 = tfidf_matrix[:n]
    tfidf_A2 = tfidf_matrix[n:2*n]
    tfidf_A3 = tfidf_matrix[2*n:3*n]

    # Compute Cosine Similarity
    df["Cosine_Sim_A1_A2"] = [cosine_similarity(tfidf_A1[i], tfidf_A2[i])[0][0] for i in range(n)]
    df["Cosine_Sim_A2_A3"] = [cosine_similarity(tfidf_A2[i], tfidf_A3[i])[0][0] for i in range(n)]
    df["Cosine_Sim_A1_A3"] = [cosine_similarity(tfidf_A1[i], tfidf_A3[i])[0][0] for i in range(n)]

    # Save to a new CSV file
    output_file = file_path.replace(".csv", "_with_cosine.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

    return df

In [ ]:
# File paths
t5_small_validated_model_answers_path = f"{DRIVE_BASE_PATH}/t5_small/modelanswersvalidated_t5small_updated.csv"
opt_validated_model_answers_path = f"{DRIVE_BASE_PATH}/opt/modelanswersvalidated_opt_updated.csv"
flan_t5_small_validated_model_answers_path = f"{DRIVE_BASE_PATH}/flan_t5_small/modelanswersvalidated_flan-t5small_updated.csv"
flan_t5_small_validated_model_answers_path_fewshot = f"{DRIVE_BASE_PATH}/flan_t5_small/modelanswersvalidated_fewshot_t5small.csv"


files = [
    t5_small_validated_model_answers_path,
    opt_validated_model_answers_path,
    flan_t5_small_validated_model_answers_path,
]

# Process each file
dfs = [compute_cosine_similarity(file) for file in files]

In [ ]:
for df in dfs:
  if df is not None:
    print(df[["Cosine_Sim_A1_A2", "Cosine_Sim_A2_A3", "Cosine_Sim_A1_A3"]].describe())

# Reshaping the Data Format in CSV Files

In [ ]:
# Function to reduce the columns and reshape the CSV files
def reshape_csv(file_path):
  # Load CSV file
  df = pd.read_csv(file_path)

  # Ensure required columns exist
  required_columns = [
      "Context", "Question", "Ground Truth Answer",
      "Answer 1", "Answer 2", "Answer 3",
      "Model Validation Answer (Answer 1)", "Model Validation Answer (Answer 2)", "Model Validation Answer (Answer 3)",
      "Human_Label1", "Human_Label2", "Human_Label3"
  ]
  missing_columns = [col for col in required_columns if col not in df.columns]
  if missing_columns:
      raise ValueError(f"Missing columns in dataset: {missing_columns}")

  # Reset index before melting to avoid lookup errors
  df = df.reset_index()

  # Melt the DataFrame to turn answers into rows
  df_long = df.melt(
      id_vars=["index", "Context", "Question", "Ground Truth Answer"],  # Keep these columns
      value_vars=["Answer 1", "Answer 2", "Answer 3"],  # Answers become rows
      var_name="Answer Type",
      value_name="Model Answer"
  )

  # Mapping dictionaries for Model Validation & Human Label
  validation_mapping = {
      "Answer 1": "Model Validation Answer (Answer 1)",
      "Answer 2": "Model Validation Answer (Answer 2)",
      "Answer 3": "Model Validation Answer (Answer 3)"
  }

  human_label_mapping = {
      "Answer 1": "Human_Label1",
      "Answer 2": "Human_Label2",
      "Answer 3": "Human_Label3"
  }

  # Merge Model Validation and Human Label columns using index
  df_long = df_long.merge(df[["index"] + list(validation_mapping.values())], on="index", how="left")
  df_long = df_long.merge(df[["index"] + list(human_label_mapping.values())], on="index", how="left")

  # Assign correct values for Model Validation & Human Label
  df_long["Model Validation"] = df_long.apply(lambda row: row[validation_mapping[row["Answer Type"]]], axis=1)
  df_long["Human Label"] = df_long.apply(lambda row: row[human_label_mapping[row["Answer Type"]]], axis=1)

  # Drop unnecessary columns
  df_long.drop(columns=["index", "Answer Type"] + list(validation_mapping.values()) + list(human_label_mapping.values()), inplace=True)

  # Save the reshaped DataFrame to a new CSV file
  output_file = file_path.replace(".csv", "_reshaped.csv")
  df_long.to_csv(output_file, index=False)

  print(f"Reshaped file saved as: {output_file}")


reshape_csv(t5_small_validated_model_answers_path)
reshape_csv(opt_validated_model_answers_path)
reshape_csv(flan_t5_small_validated_model_answers_path)
reshape_csv(flan_t5_small_validated_model_answers_path_fewshot)

t5_small_validated_model_answers_reshaped_path = f"{DRIVE_BASE_PATH}/t5_small/modelanswersvalidated_t5small_updated_reshaped.csv"
opt_validated_model_answers_reshaped_path = f"{DRIVE_BASE_PATH}/opt/modelanswersvalidated_opt_updated_reshaped.csv"
flan_t5_small_validated_model_answers_reshaped_path = f"{DRIVE_BASE_PATH}/flan_t5_small/modelanswersvalidated_flan-t5small_updated_reshaped.csv"
flan_t5_small_validated_model_answers_reshaped_path_fewshot = f"{DRIVE_BASE_PATH}/flan_t5_small/modelanswersvalidated_fewshot_t5small_reshaped.csv"

# **Accuracy of Answers Generated by the Model**

In [ ]:
# Check how many are marked as incorrect vs correct by human by model
def process_model_answers(csv_file):
    """Processes a CSV file to evaluate answers generated by the model compared to human annotation.

    - Converts 'Correct' → 0, 'Incorrect' → 1 in Model Validation & Human Label.
    - Identifies and displays rows with missing values instead of dropping them.
    - Creates a table with separate columns for correct and incorrect percentages.

    Args:
    - csv_file (str): Path to the CSV file.

    Returns:
    - DataFrame with percentage breakdowns.
    """

    print(f"Processing file: {csv_file}")

    # Load dataset
    df = pd.read_csv(csv_file)

    # Convert 'Correct' → 0, 'Incorrect' → 1
    df["Model Validation"] = df["Model Validation"].map({"correct": 0, "incorrect": 1})
    df["Human Label"] = df["Human Label"].map({"correct": 0, "incorrect": 1})

    # Identify rows with missing values
    missing_rows = df[df["Human Label"].isna() | df["Model Validation"].isna()]
    if not missing_rows.empty:
        print("Rows with missing values detected:")
        print(missing_rows)
        return missing_rows  # Return only the problematic rows for inspection

    # Compute percentages
    total_samples = len(df)
    human_correct_pct = (df["Human Label"] == 0).mean() * 100
    human_incorrect_pct = (df["Human Label"] == 1).mean() * 100
    model_correct_pct = (df["Model Validation"] == 0).mean() * 100
    model_incorrect_pct = (df["Model Validation"] == 1).mean() * 100

    # Create a structured summary table
    summary_df = pd.DataFrame({
        "Category": ["Human", "Model"],
        "Correct (%)": [human_correct_pct, model_correct_pct],
        "Incorrect (%)": [human_incorrect_pct, model_incorrect_pct]
    })

    summary_df

    return summary_df

In [ ]:
flan_t5_small_metrics_path = f"{DRIVE_BASE_PATH}/flan_t5_small/metrics_flant5.csv"
opt_metrics_path = f"{DRIVE_BASE_PATH}/opt/metrics_opt.csv"
t5_small_metrics_path = f"{DRIVE_BASE_PATH}/t5_small/metrics_t5.csv"

In [ ]:
process_model_answers(flan_t5_small_metrics_path)

In [ ]:
process_model_answers(opt_metrics_path)

In [ ]:
process_model_answers(t5_small_metrics_path)

# **Calculating BertScore & Selecting a Threshold for Hallucination Classification**

In [ ]:
# Load SentenceTransformer on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)  # Move model to GPU

In [ ]:
# Function to calculate BertScore (F1 score) for model generated answers vs the context
def evaluate_generated_answers(csv_file):
  """Evaluates answers using BERTScore."""

  df = pd.read_csv(csv_file)  # Load full CSV
  results = []

  for _, row in df.iterrows():
    context = row["Context"]  # Reference for evaluation
    model_answer = row["Model Answer"]  # Generated answer

    # Compute BERTScore (F1) on GPU
    _, _, F1 = bert_score([model_answer], [context], lang="en", device=device)

    # Store results in a dictionary
    row_results = {
        "BERTScore_F1 (Model Answer)": F1.mean().item(),
    }

    results.append(row_results)

  # Convert results into a DataFrame
  results_df = pd.DataFrame(results)

  # Merge back with original dataset to retain ALL columns
  df = df.reset_index(drop=True)  # Ensure index alignment
  results_df = results_df.reset_index(drop=True)  # Ensure index alignment
  final_df = pd.concat([df, results_df], axis=1)  # Append new columns while keeping old ones

  return final_df  # Returns the updated DataFrame with all original columns + new computed columns

In [ ]:
# Post-Fine-Tuning metrics calc
add_metrics_opt = evaluate_generated_answers(opt_validated_model_answers_reshaped_path)
add_metrics_t5 = evaluate_generated_answers(t5_small_validated_model_answers_reshaped_path)
add_metrics_flant5 = evaluate_generated_answers(flan_t5_small_validated_model_answers_reshaped_path)

# Saving all files as csv
add_metrics_opt.to_csv(opt_metrics_path)
add_metrics_t5.to_csv(t5_small_metrics_path)
add_metrics_flant5.to_csv(flan_t5_small_metrics_path)

## **Calculating a Threshold per Mdel and classifying Answers as Hallucinated vs Correct**

## Compute Mean

In [ ]:
# Compute mean BertScore for each model
def compare_models(csv_files, bert_column="BERTScore_F1 (Model Answer)"):
  """Computes the average BERTScore for each model's CSV file.

  Args:
    - csv_files (list): List of CSV file paths for different models.
    - bert_column (str): Column name for BERTScore.

  Returns:
    - DataFrame summarizing the average scores per model.
  """

  results = []

  for file in csv_files:
    df = pd.read_csv(file)

    # Check if column exists
    if bert_column not in df.columns:
      raise ValueError(f"Column '{bert_column}' not found in {file}")

    # Compute mean BERTScore
    mean_bert = df[bert_column].mean()

    results.append({"Model": file, "Avg BERTScore": mean_bert})

  # Convert results into a DataFrame for comparison
  results_df = pd.DataFrame(results)

  return results_df

In [ ]:
csv_files = [
   opt_metrics_path,
   t5_small_metrics_path,
   flan_t5_small_metrics_path
]

results_df = compare_models(csv_files)

In [ ]:
results_df.head()

## Draw Histograms to Check Distribution

In [ ]:
# Function to draw histograms for BertScores per model
def plot_bertscore_distributions(csv_files, bert_column="BERTScore_F1 (Model Answer)", bins=30):
    """Plots side-by-side histograms for BERTScore distributions across multiple models.

    Args:
    - csv_files (dict): Dictionary where keys are model names and values are file paths.
    - bert_column (str): Column name for BERTScore.
    - bins (int): Number of bins for the histogram.
    """

    fig, axes = plt.subplots(1, len(csv_files), figsize=(15, 5), sharex=True, sharey=True)

    for ax, (model, file) in zip(axes, csv_files.items()):
        df = pd.read_csv(file)

        # Ensure column exists
        if bert_column not in df.columns:
            raise ValueError(f"Column '{bert_column}' not found in {file}")

        # Plot histogram
        ax.hist(df[bert_column], bins=bins, alpha=0.7, color='blue', edgecolor='black')
        ax.axvline(x=df[bert_column].mean(), color='red', linestyle='--', label=f"Mean: {df[bert_column].mean():.3f}")
        ax.set_xlabel("BERTScore")
        ax.set_title(f"{model} BERTScore Distribution")
        ax.legend()

    axes[0].set_ylabel("Frequency")  # Set y-label only on the first plot
    plt.tight_layout()
    plt.show()


csv_files = {
    "OPT": opt_metrics_path,
    "T5": t5_small_metrics_path,
    "FLAN-T5": flan_t5_small_metrics_path,
}

plot_bertscore_distributions(csv_files)

## Apply Mean BertScore as Threshold to Classify Answers as Hallucinated vs Correct

In [ ]:
# Function to classify answers as hallucinated vs correct based on mean BertScore
def classify_hallucinations(csv_files, bert_column="BERTScore_F1 (Model Answer)", save_results=True):
  """Classifies hallucinations using a threshold for BERTScore for multiple CSV files.

  Args:
    - csv_files (list): List of CSV file paths.
    - bert_column (str): Column containing BERTScore values.
    - save_results (bool): Whether to save the processed files.

  Returns:
    - Dictionary of DataFrames with classified hallucinations.
  """

  results = {}

  for csv_file in csv_files:
    df = pd.read_csv(csv_file)

    # Ensure required column exists
    if bert_column not in df.columns:
      raise ValueError(f"Column '{bert_column}' not found in {csv_file}")

    # Compute mean BERTScore for the current file
    mean_bert = df[bert_column].mean()

    # Apply hallucination classification
    df["Hallucination_BERT(1)"] = (df[bert_column] < mean_bert).astype(int)

    # Save the updated file
    if save_results:
      output_file = csv_file.replace(".csv", "_hallucination_classified.csv")
      df.to_csv(output_file, index=False)
      print(f"✅ Processed results saved as: {output_file}")

    results[csv_file] = df

  return results


# Apply classification for each model
results = classify_hallucinations(csv_files)
metrics_opt_hallucination_classified_path = f"{DRIVE_BASE_PATH}/opt/metrics_opt_hallucination_classified.csv"
metrics_t5_hallucination_classified_path = f"{DRIVE_BASE_PATH}/t5_small/metrics_t5_hallucination_classified.csv"
metrics_flant5_hallucination_classified_path = f"{DRIVE_BASE_PATH}/flan_t5_small/metrics_flant5_hallucination_classified.csv"

In [ ]:
# Function to compare hallucination vs correct rates per model based on the BertScore classification
def compare_hallucination_results(csv_files):
    """Compares hallucination rates across multiple models using BERTScore.

    Args:
    - csv_files (dict): Dictionary where keys are model names and values are file paths.

    Returns:
    - DataFrame summarizing the percentage of hallucinations vs. non-hallucinations per model.
    """

    results = []

    for model, file in csv_files.items():
        df = pd.read_csv(file)

        # Ensure required columns exist
        if "Hallucination_BERT(1)" not in df.columns not in df.columns:
            raise ValueError(f"Columns 'Hallucination_BERT(1)' not found in {file}")

        total = len(df)

        # Compute hallucination rates for BERTScore
        bert_hallucinated_count = df["Hallucination_BERT(1)"].sum()
        bert_hallucinated_pct = (bert_hallucinated_count / total) * 100
        bert_non_hallucinated_pct = 100 - bert_hallucinated_pct


        # Store results
        results.append({
            "Model": model,
            "Total Responses": total,
            "BERT Hallucinated (%)": bert_hallucinated_pct,
            "BERT Non-Hallucinated (%)": bert_non_hallucinated_pct
        })

    # Convert results into a DataFrame for easy comparison
    results_df = pd.DataFrame(results)

    return results_df


# Define the classified hallucination result files
classified_csv_files = {
    "OPT": metrics_opt_hallucination_classified_path,
    "T5": metrics_t5_hallucination_classified_path,
    "FLAN-T5": metrics_flant5_hallucination_classified_path,
}

# Run the comparison
hallucination_comparison_df = compare_hallucination_results(classified_csv_files)
hallucination_comparison_df

In [ ]:
# Function to calculate accuracy, precision, recall, and F1-score for hallucination detection for BertScore
def process_hallucination_detection(csv_file):
    """Processes a CSV file to evaluate hallucination detection by Hallucination_BERT(1) compared to human annotation.

    - Converts 'Correct' → 0, 'Incorrect' → 1 in Human Label.
    - Uses existing binary values in Hallucination_BERT(1) (0 = correct, 1 = hallucinated).
    - Computes accuracy, precision, recall, and F1-score.
    - Generates a confusion matrix.
    - Creates a bar graph comparing human vs. model hallucination detection.
    - Displays results without saving.

    Args:
    - csv_file (str): Path to the CSV file.

    Returns:
    - DataFrame with converted labels and analysis results.
    """

    print(f"Processing file: {csv_file}")

    # Load dataset
    df = pd.read_csv(csv_file)

    # Ensure required columns exist
    if "Hallucination_BERT(1)" not in df.columns or "Human Label" not in df.columns:
        print("Error: Required columns ('Hallucination_BERT(1)', 'Human Label') not found in the CSV file.")
        return None

    # Convert 'Human Label' column: 'Correct' → 0, 'Incorrect' → 1
    df["Human Label"] = df["Human Label"].map({"correct": 0, "incorrect": 1})

    # Compute accuracy
    accuracy = accuracy_score(df["Human Label"], df["Hallucination_BERT(1)"])
    print(f"Hallucination_BERT(1) Accuracy Compared to Human Annotation: {accuracy:.4f}")

    # Compute confusion matrix
    conf_matrix = confusion_matrix(df["Human Label"], df["Hallucination_BERT(1)"])

    # Plot confusion matrix
    plt.figure(figsize=(5,5))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap="Blues", xticklabels=["Correct", "Incorrect"], yticklabels=["Correct", "Incorrect"])
    plt.xlabel("Predicted by Hallucination_BERT(1)")
    plt.ylabel("Human Annotation")
    plt.title("Confusion Matrix")
    plt.show()

    # Print classification report
    print("\nClassification Report:\n", classification_report(df["Human Label"], df["Hallucination_BERT(1)"], target_names=["Correct (0)", "Incorrect (1)"]))

    # Compute number of hallucinations detected
    human_hallucinations = df["Human Label"].sum()
    model_hallucinations = df["Hallucination_BERT(1)"].sum()

    # Bar plot comparing human vs. Hallucination_BERT(1) hallucination detection
    plt.figure(figsize=(6,4))
    plt.bar(["Human Annotation", "Hallucination_BERT(1)"], [human_hallucinations, model_hallucinations], color=["blue", "red"])
    plt.ylabel("Number of Hallucinations Detected")
    plt.title("Comparison of Hallucinations Detected by Humans vs. Hallucination_BERT(1)")
    plt.show()

    return df


In [ ]:
process_hallucination_detection(metrics_flant5_hallucination_classified_path)

In [ ]:
process_hallucination_detection(metrics_t5_hallucination_classified_path)

In [ ]:
process_hallucination_detection(metrics_opt_hallucination_classified_path)

# **Model Self-Detecting Hallucination Metrics**

In [ ]:
# Function to calculate accuracy, precision, recall and F1-score for hallucination self-detection by models
def process_hallucination_detection(csv_file):
    """Processes a CSV file to evaluate hallucination detection by a model compared to human annotation.

    - Converts 'Correct' → 0, 'Incorrect' → 1 in Model Validation & Human Label.
    - Computes accuracy.
    - Generates a confusion matrix.
    - Creates a bar graph comparing human vs. model hallucination detection.
    - Displays results in Google Colab without saving.

    Args:
    - csv_file (str): Path to the CSV file.

    Returns:
    - DataFrame with converted labels and analysis results (for display only).
    """

    print(f"Processing file: {csv_file}")

    # Load dataset
    df = pd.read_csv(csv_file)

    # Convert 'Correct' → 0, 'Incorrect' → 1
    df["Model Validation"] = df["Model Validation"].map({"correct": 0, "incorrect": 1})
    df["Human Label"] = df["Human Label"].map({"correct": 0, "incorrect": 1})

    # Compute accuracy
    accuracy = accuracy_score(df["Human Label"], df["Model Validation"])
    print(f"Model Accuracy Compared to Human Annotation: {accuracy:.4f}")

    # Display DataFrame in Colab
    #tools.display_dataframe_to_user(name="Processed Data", dataframe=df)

    # Compute confusion matrix
    conf_matrix = confusion_matrix(df["Human Label"], df["Model Validation"])

    # Plot confusion matrix
    plt.figure(figsize=(5,5))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap="Blues", xticklabels=["Correct", "Incorrect"], yticklabels=["Correct", "Incorrect"])
    plt.xlabel("Predicted by Model")
    plt.ylabel("Human Annotation")
    plt.title("Confusion Matrix")
    plt.show()

    # Print classification report
    print("\nClassification Report:\n", classification_report(df["Human Label"], df["Model Validation"], target_names=["Correct (0)", "Incorrect (1)"]))

    # Compute number of hallucinations detected
    human_hallucinations = df["Human Label"].sum()
    model_hallucinations = df["Model Validation"].sum()

    # Bar plot comparing human vs. model hallucination detection
    plt.figure(figsize=(6,4))
    plt.bar(["Human Annotation", "Model Detection"], [human_hallucinations, model_hallucinations], color=["blue", "red"])
    plt.ylabel("Number of Hallucinations Detected")
    plt.title("Comparison of Hallucinations Detected by Humans vs. Model")
    plt.show()

    return df

In [ ]:
# Running the function on multiple CSV files
for file in csv_files:
    process_hallucination_detection(file)

</br>

# **Checking Answer Length vs. Hallucination Classification by the Model Using Boxplot**

In [ ]:
# Load the three CSV files
files = [
    metrics_flant5_hallucination_classified_path,
    metrics_t5_hallucination_classified_path,
    metrics_opt_hallucination_classified_path,
]

model_names = ["FLAN-T5-Small", "T5-Small", "OPT-125M"]

# Create subplots in a row
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)  # 1 row, 3 columns

for i, file in enumerate(files):
    df = pd.read_csv(file)

    # Calculate Model Answer Length
    df['Model_Answer_Length'] = df['Model Answer'].astype(str).apply(len)

    # Generate the box plot
    sns.boxplot(x=df['Model Validation'], y=df['Model_Answer_Length'], ax=axes[i])
    axes[i].set_xlabel("Model Validation (1 = Incorrect, 0 = Correct)")
    axes[i].set_ylabel("Model Answer Length" if i == 0 else "")  # Only set y-axis label for the first plot
    axes[i].set_title(f"{model_names[i]} Model Answer Length vs. Validation")

# Show the plots
plt.tight_layout()
plt.show()

# **Computing Cohen's Kappa**

In [ ]:
# Function to compute Cohen's Kappa for a given CSV file
def compute_kappa(csv_file, human_col, model_col):
    df = pd.read_csv(csv_file)

    # Extract human labels and model predictions
    human_labels = df[human_col]
    model_labels = df[model_col]


    # Compute Cohen's Kappa
    kappa = cohen_kappa_score(human_labels, model_labels)

    return kappa

In [ ]:
# List of CSV files and column names
csv_files = [
    metrics_flant5_hallucination_classified_path,
    metrics_t5_hallucination_classified_path,
    metrics_opt_hallucination_classified_path,
]
human_col = "Human Label"  # Replace with the actual column name for human labels
model_col = "Model Validation"  # Replace with the actual column name for model predictions

In [ ]:
# Compute Cohen's Kappa for each file
for file in csv_files:
    kappa_score = compute_kappa(file, human_col, model_col)
    print(f"Cohen's Kappa for {file}: {kappa_score:.4f}")

In [ ]:
# Function to compute Cohen's Kappa after converting human labels to numeric
def compute_kappa_bert(csv_file, human_col, model_col):
    df = pd.read_csv(csv_file)

    # Convert human annotation labels to numeric format (correct → 0, incorrect → 1)
    df[human_col] = df[human_col].map({"correct": 0, "incorrect": 1})

    # Ensure model/BERTScore labels are also numeric (if not already)
    df[model_col] = df[model_col].astype(int)

    # Compute Cohen’s Kappa
    kappa = cohen_kappa_score(df[human_col], df[model_col])

    return kappa

In [ ]:
csv_files = [
    metrics_flant5_hallucination_classified_path,
    metrics_t5_hallucination_classified_path,
    metrics_opt_hallucination_classified_path,
]
human_col = "Human Label"  # Replace with the actual column name for human labels
bertscore_col = "Hallucination_BERT(1)"  # Replace with the actual column name for model predictions
# Compute Cohen's Kappa for each file
for file in csv_files:
    kappa_score = compute_kappa_bert(file, human_col, bertscore_col)
    print(f"Cohen's Kappa for {file} (BERTScore vs. Human): {kappa_score:.4f}")
